# Deliverable 3 — sistema de agentes con Qwen2.5-1.5B

Este notebook reproduce lo que muestran el video y el documento técnico:

1. corre los tests de lógica (modelo falso, sin GPU),
2. carga `Qwen/Qwen2.5-1.5B-Instruct` (el **mismo** modelo para los tres agentes),
3. evalúa B0, S2 (D2), el sistema de agentes y sus 5 ablaciones sobre **los mismos 100 estados** (50 originales + 50 nuevos del D2),
4. demo: un estado **nuevo** (semilla del reloj) con los mensajes entre agentes en vivo,
5. repite el caso de fallo y guarda su traza.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo → GPU T4`.

In [ ]:
!nvidia-smi -L

## 1. Código

In [ ]:
REPO_URL = "https://github.com/Rafaefe/AIBattlesInPokemon.git"
BRANCH = "main"      # o "d3/agent-system" mientras no este fusionada
!git clone -q -b $BRANCH $REPO_URL repo 2>/dev/null || echo 'ya estaba clonado'
%cd repo
!pip install -q -r requirements-d2.txt
import os, sys
sys.path.insert(0, os.getcwd())

## 2. Tests de lógica
Entre otras cosas: los agentes **nunca importan la tabla del juez**, la base de conocimiento dice lo mismo que el juez par por par, cada prompt es un archivo versionado y cada estado deja su traza.

In [ ]:
!python -m pytest -q tests

## 3. Cargar el modelo

In [ ]:
from d2.llm import HFModel
llm = HFModel("Qwen/Qwen2.5-1.5B-Instruct", prefix_cache=True)   # T4 -> fp32 automatico, igual que el D2

## 4. Evaluación: 100 estados × 8 sistemas
Si Colab se desconecta, vuelve a correr la celda: retoma donde quedó. Escribe `results_d3/` (decisiones, **trazas de todos los estados** y `summary.md`).

In [ ]:
from d3.evaluate import run, ALL_SYSTEMS
from d3.report import build_report
for s in ["orig", "holdout"]:
    run(llm, s, n=50, systems=ALL_SYSTEMS)
print(build_report("results_d3"))

## 5. Demo para el video: estado NUEVO
La semilla sale del reloj y se imprime; `python -m d3.demo --seed <semilla>` lo reproduce. La traza queda en `traces/`.

In [ ]:
from d3.demo import show
from d2.states import fresh_case
import time
seed = int(time.time() * 1000) % 1_000_000
show(llm, fresh_case(seed), f"seed{seed}", f"ESTADO NUEVO | semilla {seed}")

## 6. Caso de fallo
Lista los estados donde el sistema completo falla, con su causa, y repite el primero (su traza también queda en `traces/`).

In [ ]:
import json
from d3.demo import show
from d3.evaluate import SETS
from d2.states import generate_cases
summary = json.load(open("results_d3/summary.json"))
for f in summary["failures_full"]:
    print(f["case_id"], f["difficulty"], "|", f["cause"], "|", f["wrong_facts"])
if summary["failures_full"]:
    set_name, cid = summary["failures_full"][0]["case_id"].split(":")
    case = next(c for c in generate_cases(50, SETS[set_name]) if c["case_id"] == int(cid))
    show(llm, case, f"{set_name}{cid}", f"CASO DE FALLO {set_name}:{cid}")

## 7. Descargar resultados
Sube `results_d3/` y `traces/` al repositorio: de ahí salen todas las cifras del documento.

In [ ]:
!zip -qr results_d3.zip results_d3 traces
from google.colab import files
files.download("results_d3.zip")